# ETL VRA/ANAC — 2021

**ETL:** Arthur. **ELT:** Murilo Acioli. **Contrato:** v1.1.

O Python transforma os dados antes da carga. A implementação fica em
`scripts/etl_vra_anac_2021.py`; a preparação do Colab fica em `scripts/colab_etl_2021.py`.
Opcionais inválidos são auditados; os cinco campos obrigatórios determinam rejeições.

**Colab hospedado:** use uma sessão CPU nova e execute as células em ordem.
Tenha `data_2021_anac.zip` com os 12 CSVs originais, inclusive se estiverem numa pasta interna.
**Local:** use o kernel da `.venv` e os CSVs em `dados/brutos/2021/`.
As saídas anteriores foram preservadas no histórico Git e nas evidências locais.
Esta versão ainda aguarda execução real no Colab.


## 1. Preparar a sessão

No Colab, a célula clona a `main` e instala `requirements.txt`. Para testar este notebook em uma branch de revisão já publicada, altere `REF_COLAB` para o nome dessa branch na célula abaixo.
Se for solicitado reiniciar a sessão após instalar dependências, reinicie e retome do início.
Localmente, a célula apenas encontra a raiz do projeto; não instala pacotes.


In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import files
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    REF_COLAB = "main"
    RAIZ = Path("/content/Projeto-de-integracao-BD")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "--branch", REF_COLAB, "--single-branch",
                        "https://github.com/LeonardoBarrosCavalcantiNeto/Projeto-de-integracao-BD.git",
                        str(RAIZ)], check=True)
    branch = subprocess.check_output(["git", "-C", str(RAIZ), "branch", "--show-current"], text=True).strip()
    if branch != REF_COLAB:
        raise RuntimeError(f"O checkout do Colab precisa estar em {REF_COLAB}. Abra uma sessão nova.")
else:
    RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "scripts/etl_vra_anac_2021.py").is_file()), None)
    if RAIZ is None:
        raise FileNotFoundError("Abra o notebook dentro do repositório do projeto.")

for arquivo in ("requirements.txt", "scripts/etl_vra_anac_2021.py",
                "scripts/colab_etl_2021.py", "dados/manifesto_vra.json"):
    if not (RAIZ / arquivo).is_file():
        raise FileNotFoundError(f"Checkout incompleto. Arquivo ausente: {RAIZ / arquivo}")

if NO_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(RAIZ / "requirements.txt")], check=True)

sys.path.insert(0, str(RAIZ / "scripts"))
print("Ambiente:", "Colab" if NO_COLAB else "local", "| Python:", sys.version.split()[0])


## 2. Configurar e conferir o ZIP

No Colab, envie somente um ZIP. Os 12 CSVs são copiados como bytes e seus hashes
precisam coincidir com o manifesto versionado em `dados/manifesto_vra.json`. Meses ausentes, repetidos ou
arquivos diferentes interrompem a preparação, preservando arquivos locais anteriores.
Cabeçalhos e registros serão validados pelo próprio ETL, sem duplicar suas regras.


In [ ]:
import pandas as pd
from IPython.display import display
from etl_vra_anac_2021 import run_etl
from colab_etl_2021 import (
    preparar_entrada_zip, preparar_postgresql_temporario,
    validar_carga_postgresql, exportar_evidencias_colab,
)

PASTA_DADOS = RAIZ / "dados/brutos/2021"
PASTA_SAIDA = RAIZ / "dados/processados/2021/etl"
EXECUTAR_CARGA_POSTGRES = NO_COLAB
SUBSTITUIR_CARGA_POSTGRES = False
informacoes_zip = None

EXPORTAR_TRATADA_COMPLETA = False


In [ ]:
if NO_COLAB:
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError("Envie somente data_2021_anac.zip com os 12 CSVs.")
    nome_zip, conteudo_zip = next(iter(enviados.items()))
    caminho_zip = Path("/content/data_2021_anac.zip")
    caminho_zip.write_bytes(conteudo_zip)
    del enviados, conteudo_zip
    informacoes_zip = preparar_entrada_zip(
        caminho_zip, PASTA_DADOS, RAIZ / "dados/manifesto_vra.json")
    informacoes_zip["zip_enviado"] = nome_zip
    print("ZIP conferido: 12 CSVs e hashes iguais ao inventário.")
else:
    print("Entrada local:", PASTA_DADOS)


## 3. Iniciar o PostgreSQL temporário

Somente no Colab hospedado: instalar/iniciar PostgreSQL e preparar banco e usuário
`etl_2021_colab`. A senha é aleatória e fica no ambiente, sem ser exibida ou salva.
A carga usa exclusivamente `etl_2021`, sem acessar o banco do grupo.
Localmente a instalação é ignorada e a carga permanece desativada por padrão.

Em uma sessão nova, execute tudo em ordem. Se repetir a carga na mesma sessão,
ela será bloqueada. Para reconstruí-la conscientemente, configure
`SUBSTITUIR_CARGA_POSTGRES=True`; essa opção é explícita e vem desativada.


In [ ]:
if NO_COLAB:
    banco_temporario = preparar_postgresql_temporario()
    print("Banco:", banco_temporario["banco"], "| PostgreSQL:", banco_temporario["versao_postgresql"])
else:
    print("Preparação PostgreSQL ignorada no ambiente local.")


## 4. Executar o ETL

Os 20 campos brutos determinam a deduplicação; `Codeshare` permanece nos brutos.
O script padroniza textos e tipos, guarda todos os motivos de rejeição e calcula as
métricas em Python. O banco recebe os resultados já preparados.
As cópias originais e o manifesto SHA-256 ficam em `vra_bruta/` na pasta de saída.


In [ ]:
resultado = run_etl(
    input_dir=PASTA_DADOS,
    output_dir=PASTA_SAIDA,
    load_database=EXECUTAR_CARGA_POSTGRES,
    replace=SUBSTITUIR_CARGA_POSTGRES,
    schema="etl_2021",
)
print("PostgreSQL:", resultado["carga_postgresql"]["status"])

## 5. Conferir resultados e auditorias

A conciliação é **brutos = excedentes + rejeitados + tratados**, por mês e no ano.
Os totais informados pelo ELT são referência de regressão. A comparação dos valores
por registro com Murilo ainda depende de sua tabela tratada completa.


In [ ]:
display(pd.DataFrame([resultado["resumo"]]).T.rename(columns={0: "resultado"}))
display(pd.read_csv(PASTA_SAIDA / "mensal.csv"))
display(pd.read_csv(PASTA_SAIDA / "comparativo_totais_elt_informados.csv"))

In [ ]:
for nome in ["motivos_rejeicao.csv", "vra_conflitos_codigo.csv", "auditoria_esquema.csv"]:
    print(nome)
    display(pd.read_csv(PASTA_SAIDA / nome))
print("Log por arquivo")
display(pd.read_csv(PASTA_SAIDA / "etl_log_arquivos.csv"))


## 6. Conferir o banco após a carga

Uma nova transação somente de leitura confere as dez contagens, unicidade das
origens e conciliação por mês, depois do COMMIT. O resultado registra o commit
executado e a versão real do servidor. Teste local não é identificado como Colab.


In [ ]:
if EXECUTAR_CARGA_POSTGRES:
    validacao_banco = validar_carga_postgresql(resultado, RAIZ)
    display(pd.Series(validacao_banco["contagens_apos_commit"], name="registros_no_banco"))
else:
    print("Validação do banco não executada: carga desativada.")


## 7. Baixar as evidências e salvar o notebook

O ZIP reúne resumos, manifestos e auditorias. A tabela tratada completa entra
somente com `EXPORTAR_TRATADA_COMPLETA=True`. As cópias brutas não entram no ZIP;
seu manifesto fica incluído. Dados completos continuam fora do Git.

Baixe também o **notebook executado** pelo menu Arquivo → Fazer download → `.ipynb`.
O ZIP contém os quatro arquivos pequenos para revisar e colocar em `resultados/`;
as outras saídas ficam em `dados/processados/`. Confira os metadados e a regressão
antes de atualizar a documentação. Referência: 632.148 brutos = 0 excedentes +
25.511 rejeitados + 606.637 tratados. Divergências devem ser investigadas, sem filtros
para forçar esses totais. O banco pode desaparecer quando a sessão encerrar.

Depois da execução real, o terceiro commit registra notebook, README, relatório e
evidências. Esta preparação ainda não conclui a reprodução no Colab.


In [ ]:
if NO_COLAB and EXECUTAR_CARGA_POSTGRES:
    evidencias = exportar_evidencias_colab(
        resultado, informacoes_zip, incluir_tratada=EXPORTAR_TRATADA_COMPLETA)
    display(pd.DataFrame(evidencias["regressao"]).T)
    files.download(evidencias["zip"])
else:
    print("Download de evidências do Colab não executado neste ambiente.")


## Pendências

- Reproduzir esta versão no Colab e guardar as saídas reais.
- Obter a tabela tratada completa do ELT e executar o comparador.
- Resolver coletivamente os conflitos de descrição antes das dimensões e Fato.

A preparação não altera as regras do ETL nem publica dados ou credenciais no Git.
